# Evaluate PubMedBERT (PMB) on MedQA using LoRA
Files needed:

*   MedQA data (train, valid, test)

Make sure to define the path to the MedQA data

In [ ]:
path_to_train = "./train.jsonl"
path_to_valid = "./dev.jsonl"
path_to_test = "./test.jsonl"

In [ ]:
! pip3 install transformers datasets torch accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Load base data

In [ ]:
import pandas as pd
import os

train_df = pd.read_json(path_to_train, lines=True)
valid_df = pd.read_json(path_to_valid, lines=True)
test_df = pd.read_json(path_to_test, lines=True)

In [ ]:
train_df.info()
valid_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10178 entries, 0 to 10177
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    10178 non-null  object
 1   answer      10178 non-null  object
 2   options     10178 non-null  object
 3   meta_info   10178 non-null  object
 4   answer_idx  10178 non-null  object
dtypes: object(5)
memory usage: 397.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1272 entries, 0 to 1271
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    1272 non-null   object
 1   answer      1272 non-null   object
 2   options     1272 non-null   object
 3   meta_info   1272 non-null   object
 4   answer_idx  1272 non-null   object
dtypes: object(5)
memory usage: 49.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1273 entries, 0 to 1272
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtyp

In [ ]:
train_df.head()

,question,answer,options,meta_info,answer_idx
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin,"{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': '...",step2&3,E
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...,{'A': 'Placing the infant in a supine position...,step2&3,A
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud,{'A': 'Abnormal migration of ventral pancreati...,step1,A
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism,"{'A': 'Thromboembolism', 'B': 'Pulmonary ische...",step1,A
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease,"{'A': 'Factor V Leiden', 'B': 'Hemophilia A', ...",step1,E


# Now let's get serious

In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")

def preprocess_medqa(examples):
    """Tokenize question and choices separately for multiple-choice classification"""
    inputs = {"input_ids": [], "attention_mask": [], "token_type_ids": [], "labels": []}

    for example in tqdm(examples, total=len(examples)):
    #for i in range(len(examples["question"])):  # Process each example individually
        question = example["question"]
        options = example["options"] # Dictionary {'A': 'Ampicillin', 'B': 'Ceftriaxone', ...}
        correct_answer = example["answer_idx"]  # Single letter ('A', 'B', ...)

        # Ensure consistent option order (sort by key)
        option_keys = sorted(options.keys())
        option_values = [options[key] for key in option_keys]  # List of answer choices

        # Convert correct answer letter to index
        if correct_answer in option_keys:
            label = option_keys.index(correct_answer)  # Map the letter to index (0, 1, ...)
        else:
            raise ValueError(f"Unexpected answer key: {correct_answer} in {option_keys}")

        # Tokenize question-answer pairs
        encoding = tokenizer(
            [question] * len(option_values),  # Repeat the question for each choice
            option_values,  # List of choices
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # Append results to inputs dictionary
        inputs["input_ids"].append(encoding["input_ids"].squeeze(0))  # Remove extra batch dimension
        inputs["attention_mask"].append(encoding["attention_mask"].squeeze(0))
        inputs["token_type_ids"].append(encoding.get("token_type_ids", None).squeeze(0) if "token_type_ids" in encoding else None)
        inputs["labels"].append(label)  # Correct answer index

    return inputs

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

In [ ]:
import torch
from datasets import Dataset


# Convert DataFrame to Hugging Face Dataset
hf_train = Dataset.from_pandas(train_df)

# Apply tokenization
train_inputs = preprocess_medqa(hf_train)

hf_valid = Dataset.from_pandas(valid_df)
valid_inputs = preprocess_medqa(hf_valid)

# Let's try converting them back to the Dataset class
train_inputs = Dataset.from_dict(train_inputs)
valid_inputs = Dataset.from_dict(valid_inputs)



100%|██████████| 1272/1272 [00:04<00:00, 297.66it/s]


In [ ]:
from transformers import BertTokenizer, BertForMultipleChoice
from peft import get_peft_model, LoraConfig, TaskType

model = BertForMultipleChoice.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")

# Create LoRA config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # For classification tasks
    inference_mode=False,
    r=8,                              # Rank of LoRA matrices
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]  # Layer names to inject LoRA into
)

# Wrap model with LoRA
model = get_peft_model(model, peft_config)

# Make the classifying head trainable
for name, param in model.named_parameters():
    if "score" in name or "classifier" in name:  # Depending on your model architecture
        param.requires_grad = True

model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight
base_model.model.bert.encoder.layer.1.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.1.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.1.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.1.attention.self.value.lora_B.default.weight
base_model.model.bert.encoder.layer.2.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.2.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.2.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.2.attention.self.value.lora_B.default.weight
base_model.model.bert.encode

In [ ]:
# Verifying if we are working on the GPU
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # GPU name
model.to("cuda")
print(next(model.parameters()).device)  # Should return: cuda:0

True
1
Tesla T4
cuda:0


In [ ]:
import evaluate

# Load metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    return {"accuracy": accuracy["accuracy"]}

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    run_name="baseline_1.2",
    output_dir="./out/",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,  # Enable mixed precision (faster training)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    metric_for_best_model="accuracy",
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,  # Speed up data loading
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_inputs,
    eval_dataset=valid_inputs,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Pass the metric function
)

<ipython-input-11-f1678d75e29b>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.594600,1.565391,0.283805
2,1.569600,1.542300,0.298742
3,1.548100,1.527602,0.297170
4,1.521500,1.518610,0.308176
5,1.505900,1.527766,0.312107
6,1.489200,1.521472,0.309748
7,1.475200,1.528084,0.312893
8,1.444200,1.537404,0.311321
9,1.440900,1.543117,0.312107
10,1.440900,1.547646,0.312893


TrainOutput(global_step=6370, training_loss=1.4983162058017319, metrics={'train_runtime': 4849.7279, 'train_samples_per_second': 20.987, 'train_steps_per_second': 1.313, 'total_flos': 6.71791320173568e+16, 'train_loss': 1.4983162058017319, 'epoch': 10.0})

Save LoRa weights + BERT model (optional)



In [ ]:
model.save_pretrained("./lora_adapter/")

In [ ]:
!zip -r ./lora_adapter.zip ./lora_adapter/*

  adding: lora_adapter/adapter_config.json (deflated 52%)
  adding: lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: lora_adapter/README.md (deflated 65%)


In [ ]:
from google.colab import files
files.download('./lora_adapter.zip')

In [ ]:
model.base_model.save_pretrained("./base_model_with_updated_head/")
tokenizer.save_pretrained("./base_model_with_updated_head/")

('./base_model_with_updated_head/tokenizer_config.json',
 './base_model_with_updated_head/special_tokens_map.json',
 './base_model_with_updated_head/vocab.txt',
 './base_model_with_updated_head/added_tokens.json',
 './base_model_with_updated_head/tokenizer.json')

In [ ]:
!zip ./base_model_with_updated_head.zip -r ./base_model_with_updated_head/*

  adding: base_model_with_updated_head/config.json (deflated 48%)
  adding: base_model_with_updated_head/model.safetensors (deflated 7%)
  adding: base_model_with_updated_head/special_tokens_map.json (deflated 42%)
  adding: base_model_with_updated_head/tokenizer_config.json (deflated 74%)
  adding: base_model_with_updated_head/tokenizer.json (deflated 71%)
  adding: base_model_with_updated_head/vocab.txt (deflated 54%)


In [ ]:
files.download('./base_model_with_updated_head.zip')

Evaluate model on test set

In [ ]:
hf_test = Dataset.from_pandas(test_df)
test_inputs = Dataset.from_dict(preprocess_medqa(hf_test))

results = trainer.evaluate(test_inputs)
with open("test_results.csv", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")
print(results)

100%|██████████| 1273/1273 [00:04<00:00, 287.95it/s]


{'eval_loss': 1.5203543901443481, 'eval_accuracy': 0.32050274941084056, 'eval_runtime': 23.3836, 'eval_samples_per_second': 54.44, 'eval_steps_per_second': 3.421, 'epoch': 10.0}
